In [27]:
import os
import gzip
import json
import pandas as pd
from pathlib import Path
from collections.abc import Iterable, Iterator

# for agent
from src.em_agent.graph import graph
from src.em_agent.context import AnalysisContext
from src.em_agent.tracing import langfuse_handler

# env variable
from dotenv import load_dotenv
load_dotenv()

True

In [28]:
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "archive").is_dir())
ARCHIVE_DIR = REPO / "data" / "archive"  # gitignored download cache

In [29]:
def _resolve_paths(source: str | Path | Iterable[str | Path]) -> list[Path]:
    if isinstance(source, (str, Path)):
        p = Path(source)
        if p.is_dir():
            return sorted(p.glob("*.jsonl.gz"))
        return [p]
    return [Path(p) for p in source]

def _parse_filename(path: Path) -> tuple[str, str]:
    """``EARNINGS_RELEASE_2025Q3.jsonl.gz`` → ``("EARNINGS_RELEASE", "2025Q3")``."""
    stem = path.name.removesuffix(".jsonl.gz")
    event_type, _, quarter = stem.rpartition("_")
    return event_type, quarter

def read_jsonl_gz(path: str | Path) -> Iterator[dict]:
    """Yield each record from a gzip-JSONL file, skipping blank lines."""
    with gzip.open(path, "rt", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                yield json.loads(line)

def _load_one(path: Path) -> pd.DataFrame:
    df = pd.DataFrame(read_jsonl_gz(path))
    event_type, quarter = _parse_filename(path)
    # Add provenance from the filename, but never clobber a field the record
    # already carries. Insert at the front so it reads left-to-right.
    if "quarter" not in df.columns:
        df.insert(0, "quarter", quarter)
    if "event_type" not in df.columns:
        df.insert(0, "event_type", event_type)
    return df

def load_archive(source: str | Path | Iterable[str | Path]) -> pd.DataFrame:
    """Load one or more ``*.jsonl.gz`` archive files into a single DataFrame.

    ``source`` may be a directory (all ``*.jsonl.gz`` within it), a single file,
    or an iterable of paths. Provenance columns ``event_type`` and ``quarter``
    are added from each filename — unless a record already carries that field, in
    which case the record's own value is kept.
    """
    frames = [_load_one(p) for p in _resolve_paths(source)]
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

In [30]:
df_archive = load_archive(ARCHIVE_DIR)

In [31]:
def safe_get(data, *keys):
    for key in keys:
        if not isinstance(data, dict):
            return None
        data = data.get(key)
    return data

def form_ctx(event_index = 8500):
    row = df_archive.iloc[event_index]
    event_id = row["event_id"]
    event_type = row["event_type"]
    timing_category = row["timing_category"]
    event_datetime = row["event_datetime"]
    knowledge_cutoff = row["knowledge_cutoff"]
    focal_assets = row["focal_assets"]
    identifier_type = focal_assets[0]["identifier_type"]
    identifier_value = focal_assets[0]["identifier_value"]

    disclosure = row.get("disclosure") or {}
    items = {
        d["id"]: d.get("content")
        for d in (disclosure.get("items") or [])
        if isinstance(d, dict) and d.get("id") is not None
    }

    earnings_call_date = disclosure.get("generated_at")
    earnings_call_facts = items.get("earnings-call-facts")
    earnings_preview = items.get("earnings-preview")
    option_stats = items.get("option-implied-stats")

    implied_earnings_volatility = safe_get(
        option_stats, "implied_earnings_volatility", "value"
    )

    implied_absolute_earnings_move = safe_get(
        option_stats, "implied_absolute_earnings_move", "value"
    )

    skew_25_delta = safe_get(
        option_stats, "skew_25_delta", "value"
    )

    ctx = AnalysisContext(
        stock_id = identifier_value,
        earnings_call_date = earnings_call_date,
        earnings_call_facts = earnings_call_facts,
        earnings_preview = earnings_preview,
        option_stats = option_stats,
        knowledge_cutoff = knowledge_cutoff,
        num_rounds = 2
    )
    return ctx

In [ ]:
ctx = form_ctx(8500)
# result = graph.invoke({}, context=ctx, config={"callbacks": [langfuse_handler]})

In [26]:
df_archive.metrics[0]

{'earnings_surprise': {'surprise': 0.0002160760587726,
  'surprise_status': 'ok',
  'price_date': '2025-10-08'}}

In [18]:
import yfinance as yf

data = yf.download("MDB", period="1mo")
display(data)

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,MDB,MDB,MDB,MDB,MDB
Date,,,,,
2026-09-22,427.690002,439.750000,406.660004,409.309998,2113100
2026-09-23,428.359985,438.278015,424.140015,429.339996,1192100
2026-09-24,421.399994,427.000000,416.390015,423.040009,1204600
2026-09-25,410.440002,432.859985,409.179993,417.600006,1381100
2026-09-28,334.679993,344.302002,300.000000,311.345001,17855100
2026-09-29,337.190002,349.000000,321.000000,335.144989,9347900
2026-09-30,348.609985,370.429993,339.929993,341.000000,6258500
2026-10-01,351.730011,355.649994,342.000000,352.470001,2622900
